# `evaluate` — Honest Signal Evaluation

The whole point of the framework: measure a signal the way a sceptical PM would, not the way that flatters it. Everything below is **leakage-safe** — the signal known at the *close of day t* is paired with the return earned over day *t+1* (`returns.shift(-1)`). We never let a signal 'predict' the same day's return.

### What we measure
* **Rank IC (Spearman)** — daily rank correlation of signal vs next-day return. We report the mean IC, its **t-stat** $=\overline{IC}/\sigma_{IC}\times\sqrt{N}$ (is the edge real?), and the hit-rate.
* **IC decay** — how fast the edge fades at horizons t+1…t+10.
* **Turnover** — the fraction of the long-short book rebuilt each day; this is what turns a paper edge into a cost bill.
* **Quintile long-short spread** — top quintile minus bottom, reported **gross** and **net of costs**.
* **Fundamental Law** — $IR \approx IC \times \sqrt{BR}$ (note the square root on breadth).

The non-obvious mechanics (the IC-decay shift trick, the turnover formula, the cost charge) are explained in the `#` comments below.

In [ ]:
%run config.ipynb

In [ ]:
"""Signal evaluation: IC statistics, IC decay, turnover, and gross/net quantile P&L."""
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from typing import Dict, Tuple


def _daily_rank_ic(signal: pd.DataFrame, fwd: pd.DataFrame, idx) -> pd.Series:
    """Spearman rank IC between the signal and forward returns, one number per day."""
    ic = pd.Series(np.nan, index=idx)
    for date in idx:
        s, r = signal.loc[date], fwd.loc[date]
        v = s.notna() & r.notna()   # a stock counts only if it has BOTH a signal and a return
        # Why Spearman (rank correlation) rather than Pearson?
        #  1. We only care about ORDERING -- did the stocks I ranked highest actually outperform?
        #     We are not claiming the relationship is linear, and we never need it to be.
        #  2. Daily stock returns have fat tails. A single huge move would dominate a Pearson
        #     correlation; converting to ranks caps the influence of any one observation.
        # The `> 5` guard skips days too sparse for a correlation to mean anything -- with a
        # handful of names the estimate is almost pure noise.
        if v.sum() > 5:
            ic.loc[date] = spearmanr(s[v], r[v]).correlation
    return ic.dropna()


def compute_quantiles_and_turnover(signal: pd.DataFrame,
                                   forward_returns: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    """Sort each day into quintiles; return per-quintile returns and the daily turnover series."""
    q = config.QUANTILES
    common_idx = signal.index.intersection(forward_returns.index)
    q_returns = pd.DataFrame(0.0, index=common_idx, columns=range(1, q + 1))
    weights_history = []   # remember each day's weights so we can measure how much we traded

    for date in common_idx:
        sig_row, ret_row = signal.loc[date], forward_returns.loc[date]
        valid = sig_row.notna() & ret_row.notna()
        if valid.sum() < q:                       # too few names to fill q buckets -> sit out
            weights_history.append(pd.Series(0.0, index=signal.columns))
            continue

        # Rank the stocks, then cut into q equally-populated buckets.
        # Why rank first rather than cutting the raw signal values? qcut splits on quantiles of
        # the data, so if several stocks share an identical value the computed bin edges can
        # collide and pandas raises "duplicate bin edges". rank(method='first') breaks ties by
        # position, guaranteeing strictly increasing values and therefore q clean buckets.
        # Sorting into buckets (rather than weighting by the signal value) is also deliberate:
        # it is a non-parametric view that does not assume the signal is linearly related to
        # returns, and it lets us check MONOTONICITY -- does each bucket beat the one below it?
        ranks = sig_row[valid].rank(method='first')
        labels = pd.qcut(ranks, q=q, labels=range(1, q + 1))

        w = pd.Series(0.0, index=signal.columns)
        for bucket in range(1, q + 1):
            members = labels[labels == bucket].index
            q_returns.loc[date, bucket] = ret_row[members].mean()   # bucket's next-day return
            # Build the long-short book: equal-weight long the top bucket, short the bottom.
            # Equal weights (rather than signal-proportional) keep this a clean test of the
            # RANKING itself. Longs sum to +1 and shorts to -1, so the book is dollar-neutral:
            # its P&L comes from the top bucket beating the bottom one, not from the market
            # going up -- which is the whole point of a market-neutral study.
            if bucket == q:
                w[members] = 1.0 / len(members)
            elif bucket == 1:
                w[members] = -1.0 / len(members)
        weights_history.append(w)

    # Turnover = how much of the book we have to rebuild today.
    # diff() gives each stock's weight change vs yesterday; summing the ABSOLUTE changes counts
    # the selling and the buying together, so we halve it to express turnover as a one-sided
    # fraction of capital traded. Example: sell 10% of the book and buy 10% somewhere else ->
    # the absolute changes sum to 0.20 -> turnover = 10%. This single number is the bridge
    # between a statistical edge and a real trading bill.
    w_df = pd.DataFrame(weights_history, index=common_idx)
    turnover_series = 0.5 * w_df.diff().abs().sum(axis=1)
    return q_returns, turnover_series


def evaluate_signal_performance(signal: pd.DataFrame, returns: pd.DataFrame) -> Dict:
    """Run the full evaluation suite for one signal. Leakage-safe: signal_t vs return_{t+1}."""
    # *** THE LEAKAGE GUARD ***
    # shift(-1) pulls tomorrow's return back onto today's row, so row t ends up holding
    # (signal known at the close of day t, return earned over day t+1). Without this we would be
    # correlating a signal with the SAME day's return -- "predicting" something that had already
    # happened. That is the single most common way a backtest fools its own author, and it is
    # why every metric below is built on `forward_returns` rather than on `returns`.
    forward_returns = returns.shift(-1)
    common_idx = (signal.dropna(how='all').index
                        .intersection(forward_returns.dropna(how='all').index))

    # --- 1) The daily rank IC series, and whether it is distinguishable from luck ---
    ic_series = _daily_rank_ic(signal, forward_returns, common_idx)
    mean_ic = ic_series.mean()
    n_days = len(ic_series)
    ic_std = ic_series.std()

    # This t-stat is just a ONE-SAMPLE T-TEST on the daily IC series, testing
    #     H0: true mean IC = 0   (the signal has no skill whatsoever)
    # The standard error of a sample mean is s/sqrt(N), so
    #     t = mean / (s / sqrt(N)) = mean/s * sqrt(N)
    # which is exactly the line below. |t| > ~2 is the conventional bar for "unlikely to be
    # luck". Notice WHAT makes t large: a big IC helps, but so does a STABLE IC (small s) and a
    # long sample (large N). Consistency matters as much as magnitude -- which is why a tiny
    # mean IC of 0.02 can still be a real, usable edge if it shows up day after day.
    #
    # Honest caveat: this formula assumes the daily ICs are independent draws. At the 1-day
    # horizon that is roughly fine, but for the multi-day horizons further down the forward
    # windows OVERLAP, which induces positive autocorrelation and would make a naive t-stat too
    # optimistic. (The standard fix is a Newey-West/HAC standard error; we sidestep it by only
    # reporting the mean IC for those horizons, never a t-stat.)
    ic_t_stat = (mean_ic / ic_std * np.sqrt(n_days)) if ic_std > 0 else 0.0

    # Hit rate = the fraction of days the IC came out positive. Think of it as a sign test:
    # under the null of no skill you would expect about 50%. It is a robustness check on the
    # mean IC, which a handful of extreme days could otherwise flatter -- a signal with a good
    # mean but a ~50% hit rate is winning rarely and largely, which is far more fragile.
    hit_rate = (ic_series > 0).sum() / n_days if n_days > 0 else 0.0

    # --- 2) Fundamental Law scale check: IR ~= IC * sqrt(breadth) ---
    # Skill per bet, amplified by the number of INDEPENDENT bets. The square root is the part
    # people forget: doubling breadth multiplies IR by only ~1.41, and no amount of breadth can
    # rescue a signal whose net IC is negative. We treat this as a sanity-check on scale, not as
    # a tradeable Sharpe -- BR here is a hand-set constant, not a measured quantity.
    fundamental_ir = mean_ic * np.sqrt(config.ESTIMATED_BREADTH)

    # --- 3) IC decay: how long does the edge actually last? ---
    decay = {}
    for h in config.DECAY_HORIZONS:
        # We want the CUMULATIVE return over the next h days -- the window (t, t+h] -- lined up
        # on row t. Two steps get us there:
        #     .shift(-h)         -> row t now holds r_{t+h}
        #     .rolling(h).sum()  -> row t sums the h values ending at r_{t+h},
        #                           i.e. r_{t+1} + r_{t+2} + ... + r_{t+h}
        # Sanity check: at h=1 this collapses to plain r_{t+1}, matching the headline IC above.
        # (Summing simple returns slightly understates true compounding, but rank correlation
        # only cares about ordering and the sum is monotone in the compounded return.)
        fwd_h = returns.shift(-h).rolling(window=h).sum()
        ic_h = _daily_rank_ic(signal, fwd_h, common_idx)
        # Read the resulting curve like this: a signal whose IC collapses toward zero within a
        # few days must be re-traded constantly to be harvested, which is exactly what makes it
        # expensive. A signal whose IC holds up is one you can hold -- cheap to run.
        decay[f'Horizon_{h}'] = ic_h.mean() if len(ic_h) else 0.0

    # --- 4) Quintile P&L, turnover, and the gross-vs-net story ---
    q_returns, turnover_series = compute_quantiles_and_turnover(signal, forward_returns)
    avg_turnover = turnover_series.mean()
    ls_gross = q_returns[config.QUANTILES] - q_returns[1]   # top quintile minus bottom quintile

    # Charge the cost where the trading actually happens: each day's drag is THAT day's turnover
    # times the per-unit cost. Using a single average turnover for the whole sample would smear
    # the cost evenly and hide the fact that a fast signal bleeds on every single rebalance.
    # 5 bps = 5/10,000 = 0.0005 of the notional traded.
    turn_aligned = turnover_series.reindex(ls_gross.index).fillna(0.0)
    cost = turn_aligned * (config.TRANSACTION_COST_BPS / 1e4)
    ls_net = ls_gross - cost

    # Cost sensitivity. The headline 5 bps is an assumption, not a measurement, so the honest
    # thing is to show how the verdict moves as that assumption changes. Re-price the same book
    # at several cost levels and record the annualized net return at each.
    # This also answers the obvious challenge -- "what if your cost number is wrong?" -- with a
    # number instead of a shrug: a signal that only works at 1 bp is not a real signal.
    net_by_bps = {
        bps: (ls_gross - turn_aligned * (bps / 1e4)).mean() * config.TRADING_DAYS_PER_YEAR
        for bps in config.COST_SENSITIVITY_BPS
    }

    # Breakeven cost: the bps level at which the book's edge is exactly consumed. Solve
    # mean(gross) = bps/1e4 * mean(turnover) for bps. A large breakeven means a comfortable
    # margin of safety; a breakeven below realistic costs means the signal is untradeable.
    mean_turn = turn_aligned.mean()
    breakeven_bps = (ls_gross.mean() / mean_turn * 1e4) if mean_turn > 0 else float('nan')

    # Compound the daily long-short returns into equity curves for the plots. cumprod (rather
    # than a running sum) is what makes these paths comparable to a real account balance.
    cum_gross = (1 + ls_gross.fillna(0)).cumprod() - 1
    cum_net = (1 + ls_net.fillna(0)).cumprod() - 1

    return {
        'mean_ic': mean_ic,
        'ic_t_stat': ic_t_stat,
        'hit_rate': hit_rate,
        'fundamental_ir': fundamental_ir,
        'ic_decay': decay,
        'avg_turnover': avg_turnover,
        'net_by_bps': net_by_bps,
        'breakeven_bps': breakeven_bps,
        # Annualize by multiplying the mean DAILY return by 252 trading days. This is the simple
        # (arithmetic) annualization -- it ignores compounding, so treat it as a scale figure for
        # comparing the two signals rather than as a promised yearly return.
        'gross_ann_return': ls_gross.mean() * config.TRADING_DAYS_PER_YEAR,
        'annualized_net_return': ls_net.mean() * config.TRADING_DAYS_PER_YEAR,
        'cum_ic_curve': ic_series.cumsum(),
        'quantile_returns': q_returns.mean() * config.TRADING_DAYS_PER_YEAR,
        'equity_curve_gross': cum_gross,
        'equity_curve_net': cum_net,
    }


print("evaluation helpers ready: evaluate_signal_performance(), compute_quantiles_and_turnover()")